# Pasarela Colab para motor OCR existente\nNo instala el motor original. Requiere un backend OCR activo y accesible desde Colab.\nEjecuta las celdas en orden y después configura el cliente Windows.

In [ ]:
!pip -q install fastapi uvicorn httpx python-multipart
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null


In [ ]:
import os, secrets
from getpass import getpass
# Usa una clave nueva para el cliente -> Colab; no la publiques.
os.environ["OCR_API_KEY"] = getpass("Clave nueva para el cliente OCR: ").strip()
# URL del MOTOR EXISTENTE, por ejemplo http://servidor:10001
os.environ["OCR_BACKEND_URL"] = input("URL del motor OCR activo: ").strip().rstrip("/")
os.environ["OCR_BACKEND_API_KEY"] = getpass("Clave del motor OCR existente: ").strip()
assert os.environ["OCR_API_KEY"] and os.environ["OCR_BACKEND_URL"]


In [ ]:
from google.colab import files
print("Sube el archivo colab_proxy.py incluido en el ZIP")
files.upload()


In [ ]:
import subprocess,time,re,os
api=subprocess.Popen(["uvicorn","colab_proxy:app","--host","127.0.0.1","--port","8000"],stdout=open('/tmp/ocr_api.log','w'),stderr=subprocess.STDOUT)
time.sleep(3)
tunnel_log=open('/tmp/ocr_tunnel.log','w')
tunnel=subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
url=None
for _ in range(30):
    time.sleep(1)
    txt=open('/tmp/ocr_tunnel.log').read()
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',txt)
    if m: url=m.group(0);break
if not url: print('No se pudo obtener URL. Revisa /tmp/ocr_tunnel.log')
else: print('COLAB_OCR_URL='+url+'\nCopia esta URL en tu .env local. Mantén la sesión de Colab abierta.')
